# 응용 모의고사 Set 2 — 정답 — 서비스 전처리와 이탈 예측

- 데이터: `galaxy_users.csv`
- 난이도: 기존 Set 01~06과 유사
- 구성: **공통 전처리 → Q1 통계 → Q2 상관분석 → Q3 모델링**
- 모든 문항은 공통 전처리 결과를 이어서 사용합니다.
- 전처리 완료 후 데이터는 **5,512행**이어야 합니다. 행 수가 다르면 다음 문제로 넘어가기 전에 전처리를 확인하세요.

정답 노트북은 `../answers/`에 있습니다.

## 공통 전처리 정답

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv('../../dataset/galaxy_users.csv')
base = df.copy()
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']
valid = base[service_cols].isin(['Yes', 'No']).all(axis=1)
base = base.loc[valid].copy()
base[service_cols] = base[service_cols].replace({'Yes': 1, 'No': 0})
base['service_count'] = base[service_cols].sum(axis=1)
base['used_month'] = base['tenure'] // 12
binary_cols = ['Partner', 'Dependents', 'PaperlessBilling', 'Churn']
base[binary_cols] = base[binary_cols].replace({'Yes': 1, 'No': 0})
assert len(base) == 5512
display(base.head())

## Q1 정답

In [ ]:
many = (base['service_count'] >= 4).sum()
few = (base['service_count'] <= 1).sum()
answer_q1 = round(many / few, 2)
display(answer_q1)  # 1.03

## Q2 정답

In [ ]:
cols = ['tenure', 'MonthlyCharges', 'used_month', 'TotalCharges']
corr_abs = base[cols].corr(method='pearson').abs()
np.fill_diagonal(corr_abs.values, 0)
answer_q2 = round(corr_abs.max().max(), 3)
display(corr_abs, answer_q2)  # 0.989

## Q3 정답

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

features = ['SeniorCitizen', 'Partner', 'Dependents', 'tenure',
            'MonthlyCharges', 'TotalCharges', 'service_count', 'PaperlessBilling']
X = base[features]
y = base['Churn']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=321, stratify=y
)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)
pred = model.predict(X_test_scaled)
answer_q3 = round(f1_score(y_test, pred), 2)
display(answer_q3)  # 0.58